# Per-File Feature Importance — Wood Powder Segmentation

Standalone notebook: loads every `.las` file directly, extracts the full geometric
feature set from scratch (same pipeline as the main PTv2 notebook), trains a
**separate RandomForest per file**, and shows feature-importance **per file** in
one comparison figure — so you can see whether the important features are
consistent across files or vary from file to file.

This is a fast diagnostic tool (minutes, not hours) — it does not train or
evaluate PTv2 itself.

In [1]:
import os, glob, io
import numpy as np
import laspy
import open3d as o3d
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd
import logging

logging.basicConfig(level=logging.WARNING)
log = logging.getLogger("feat_importance")
RAM_STORE = {}

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Config — point this at your `.las` folder

In [2]:
CONFIG = {
    "data_dir"        : "data",          # <-- folder containing your .las files (searched recursively)
    "voxel_size"      : 0.02,
    "sor_k"           : 16,
    "sor_std"         : 2.0,
    "cluster_eps"     : 0.03,
    "cluster_min_size": 100,
    "geom_radius"     : 0.06,
    "density_k"       : 16,
    "target_class"    : 1,
    "rf_n_estimators" : 150,
    "rf_max_depth"    : 14,
    "test_size"       : 0.25,
    "min_points_after_preprocessing": 500,  # skip files that end up too small/empty
}

ALL_FILES = sorted(glob.glob(os.path.join(CONFIG["data_dir"], "**", "*.las"), recursive=True))
_data_dir = CONFIG["data_dir"]
assert ALL_FILES, f"No .las files found under '{_data_dir}' -- update CONFIG['data_dir']"
print(f"Found {len(ALL_FILES)} .las files")

Found 116 .las files


## Feature extraction (same geometric pipeline as the main PTv2 notebook)

In [3]:
def load_pointcloud(path, return_rgb=False):
    if path in RAM_STORE:
        las = laspy.read(io.BytesIO(RAM_STORE[path]))
    else:
        las = laspy.read(path)
    pts = np.column_stack([np.asarray(las.x),
                           np.asarray(las.y),
                           np.asarray(las.z)]).astype(np.float64)
    labels = None
    for key in ("classification", "label", "labels", "class"):
        if key in las.point_format.dimension_names:
            labels = np.asarray(getattr(las, key), dtype=np.int64)
            break
    rgb = None
    if return_rgb:
        dims = las.point_format.dimension_names
        if all(k in dims for k in ("red", "green", "blue")):
            rgb = np.column_stack([np.asarray(las.red),
                                   np.asarray(las.green),
                                   np.asarray(las.blue)]).astype(np.float32)
            if rgb.max() > 255:
                rgb /= 65535.0
            elif rgb.max() > 1:
                rgb /= 255.0
    finite = np.isfinite(pts).all(axis=1)
    if not finite.all():
        n_bad = int((~finite).sum())
        log.warning(f"{os.path.basename(path)}: dropping {n_bad} non-finite points")
        pts = pts[finite]
        if labels is not None:
            labels = labels[finite]
        if rgb is not None:
            rgb = rgb[finite]
    if labels is not None:
        assert len(labels) == len(pts), \
            f"{os.path.basename(path)}: label/point count mismatch"
    if return_rgb:
        return pts, rgb, labels
    return pts, labels



In [4]:
def _geom_features(points, radius=0.06, min_neighbors=8,
                   want_linear=True, want_planar=False, want_sphere=False,
                   want_vert=False, want_entropy=False,
                   want_height_std=False, want_height_range=False,
                   want_curvature=False, want_roughness=False):
    from scipy.spatial import cKDTree as _GeomTree
    n = len(points)
    lin    = np.zeros(n, np.float32) if want_linear else None
    pla    = np.zeros(n, np.float32) if want_planar else None
    sph    = np.zeros(n, np.float32) if want_sphere else None
    vert   = np.zeros(n, np.float32) if want_vert else None
    ent    = np.zeros(n, np.float32) if want_entropy else None
    hstd   = np.zeros(n, np.float32) if want_height_std else None
    hrange = np.zeros(n, np.float32) if want_height_range else None
    curv   = np.zeros(n, np.float32) if want_curvature else None
    rough  = np.zeros(n, np.float32) if want_roughness else None
    if not (want_linear or want_planar or want_sphere or want_vert or want_entropy
            or want_height_std or want_height_range or want_curvature or want_roughness):
        return lin, pla, sph, vert, ent, hstd, hrange, curv, rough
    need_evecs = want_vert or want_roughness
    tree = _GeomTree(points)
    lists = tree.query_ball_point(points, r=radius, workers=-1)
    EIG_FLOOR = 1e-6  # prevent ratio-explosion
    for i, nbrs in enumerate(lists):
        if len(nbrs) < min_neighbors:
            continue
        nb = points[nbrs]
        if want_height_std or want_height_range:
            nb_z = nb[:, 2]
            if want_height_std:   hstd[i] = nb_z.std()
            if want_height_range: hrange[i] = nb_z.max() - nb_z.min()
        centroid = nb.mean(0)
        c = nb - centroid
        cov = (c.T @ c) / len(nbrs)
        if need_evecs:
            evals, evecs = np.linalg.eigh(cov)
            l3, l2, l1 = evals
        else:
            l3, l2, l1 = np.linalg.eigvalsh(cov)
        if l1 < EIG_FLOOR:
            continue
        l1c = l1
        if want_linear:    lin[i] = (l1 - l2) / l1c
        if want_planar:    pla[i] = (l2 - l3) / l1c
        if want_sphere:    sph[i] = l3 / l1c
        if want_curvature:
            s_sum = l1 + l2 + l3
            curv[i] = l3 / max(s_sum, 1e-9)
        if want_vert or want_roughness:
            normal_proxy = evecs[:, 0]
            if want_vert:    vert[i] = 1.0 - abs(normal_proxy[2])
            if want_roughness: rough[i] = abs((points[i] - centroid) @ normal_proxy)
        if want_entropy:
            s = l1 + l2 + l3
            if s > 1e-9:
                p = np.clip(np.array([l1, l2, l3]) / s, 1e-12, None)
                ent[i] = -(p * np.log(p)).sum()
    return lin, pla, sph, vert, ent, hstd, hrange, curv, rough
def _density_features(points, k=16, want_density=False, want_mean_distance=False):
    from scipy.spatial import cKDTree as _DensTree
    n = len(points)
    density = np.zeros(n, np.float32) if want_density else None
    mean_distance = np.zeros(n, np.float32) if want_mean_distance else None
    if not (want_density or want_mean_distance) or n < 2:
        return density, mean_distance
    tree = _DensTree(points)
    k_eff = min(k, n - 1)
    dist_k, _ = tree.query(points, k=k_eff + 1)
    dist_k = dist_k[:, 1:]
    if want_mean_distance:
        mean_distance[:] = dist_k.mean(axis=1)
    if want_density:
        local_radius = dist_k.max(axis=1).clip(min=1e-9)
        raw_density = k_eff / ((4 / 3) * np.pi * local_radius ** 3)
        density[:] = np.log1p(raw_density)
    return density, mean_distance
def make_features(points, rgb=None):
    center = points.mean(axis=0, keepdims=True)
    scale  = max(np.linalg.norm(points - center, axis=1).max(), 1e-9)
    norm_xyz = ((points - center) / scale).astype(np.float32)
    z = points[:, 2]
    height = ((z - z.min()) / max(z.max() - z.min(), 1e-6)).astype(np.float32)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(
        radius=CONFIG.get("normal_radius", 0.05),
        max_nn=CONFIG.get("normal_max_nn", 30)))
    pcd.orient_normals_to_align_with_direction([0., 0., 1.])
    normals = np.asarray(pcd.normals, dtype=np.float32)
    cols = [norm_xyz, height[:, None], normals]
    want_lin    = CONFIG.get("use_linearity", False)
    want_pla    = CONFIG.get("use_planarity", False)
    want_sph    = CONFIG.get("use_sphericity", False)
    want_vert   = CONFIG.get("use_verticality", False)
    want_ent    = CONFIG.get("use_eigenentropy", False)
    want_hstd   = CONFIG.get("use_height_std", False)
    want_hrange = CONFIG.get("use_height_range", False)
    want_curv   = CONFIG.get("use_curvature", False)
    want_rough  = CONFIG.get("use_roughness", False)
    if any([want_lin, want_pla, want_sph, want_vert, want_ent,
           want_hstd, want_hrange, want_curv, want_rough]):
        lin, pla, sph, vert, ent, hstd, hrange, curv, rough = _geom_features(
            points, radius=CONFIG.get("geom_radius", 0.06),
            want_linear=want_lin, want_planar=want_pla, want_sphere=want_sph,
            want_vert=want_vert, want_entropy=want_ent,
            want_height_std=want_hstd, want_height_range=want_hrange,
            want_curvature=want_curv, want_roughness=want_rough)
        if want_lin:    cols.append(lin[:, None])
        if want_pla:    cols.append(pla[:, None])
        if want_sph:    cols.append(sph[:, None])
        if want_vert:   cols.append(vert[:, None])
        if want_ent:    cols.append(ent[:, None])
        if want_hstd:   cols.append(hstd[:, None])
        if want_hrange: cols.append(hrange[:, None])
        if want_curv:   cols.append(curv[:, None])
        if want_rough:  cols.append(rough[:, None])
    want_density = CONFIG.get("use_density", False)
    want_mdist   = CONFIG.get("use_mean_distance", False)
    if want_density or want_mdist:
        dens, mdist = _density_features(
            points, k=CONFIG.get("density_k", 16),
            want_density=want_density, want_mean_distance=want_mdist)
        if want_density: cols.append(dens[:, None])
        if want_mdist:   cols.append(mdist[:, None])
    if CONFIG.get("use_rgb", False):
        if rgb is None:
            log.warning("use_rgb=True but file has no RGB — filling zeros")
            rgb = np.zeros((len(points), 3), np.float32)
        cols.append(rgb.astype(np.float32))
    feat = np.column_stack(cols)
    assert feat.shape[1] == CONFIG["in_channels"], \
        f"feature dims {feat.shape[1]} != CONFIG in_channels {CONFIG['in_channels']}"
    # Final safety net: no single feature channel should ever be able to
    # overflow float16 (AMP) regardless of which specific computation
    # produced it. This clips the ENTIRE assembled feature array to a
    # generous but safe range -- xyz/height/normals/ratio-based geometric
    # features are all naturally well within +-10, so this only ever
    # affects a channel that has genuinely gone wrong (e.g. an unbounded
    # density-like statistic), acting as a last line of defense on top of
    # any per-feature fix (like the log1p on density).
    feat = np.clip(feat, -100.0, 100.0)
    if not np.isfinite(feat).all():
        log.error("  make_features: NaN/Inf survived even after clipping -- "
                  "sanitizing to 0.0 (this should not normally happen)")
        feat = np.nan_to_num(feat, nan=0.0, posinf=100.0, neginf=-100.0)
    return feat
def _voxel_keep(pts, voxel):
    vox = np.floor(pts / voxel).astype(np.int64)
    _, inv, cnt = np.unique(vox, axis=0, return_inverse=True, return_counts=True)
    sums = np.zeros((cnt.size, 3), np.float64); np.add.at(sums, inv, pts)
    d2 = ((pts - (sums / cnt[:, None])[inv]) ** 2).sum(1)
    order = np.lexsort((d2, inv))
    first = np.concatenate([[0], np.cumsum(cnt)[:-1]])
    return np.sort(order[first])
def _sor_keep(pts, k=16, std_ratio=2.0):
    from scipy.spatial import cKDTree as _SORTree
    n = len(pts)
    if n <= k + 1:
        return np.ones(n, bool)
    d, _ = _SORTree(pts).query(pts, k=k + 1)
    mean_dist = d[:, 1:].mean(axis=1)
    thr = mean_dist.mean() + std_ratio * mean_dist.std()
    return mean_dist <= thr
def _cluster_keep(pts, eps, min_samples=8, min_cluster_size=100):
    from sklearn.cluster import DBSCAN as _DBSCAN
    if len(pts) < min_samples:
        return np.ones(len(pts), bool)
    labels = _DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit_predict(pts)
    keep = np.zeros(len(pts), bool)
    for lab in np.unique(labels):
        if lab == -1:
            continue
        idx = np.flatnonzero(labels == lab)
        if len(idx) >= min_cluster_size:
            keep[idx] = True
    return keep
FEATURE_CACHE = {}
PREPROCESS_CACHE_DIR = CONFIG.get("preprocess_cache_dir", "data/preprocessed_cache")
os.makedirs(PREPROCESS_CACHE_DIR, exist_ok=True)


In [5]:
FEATURE_NAMES = [
    "x", "y", "z", "height",
    "linearity", "planarity", "sphericity", "verticality", "eigenentropy",
    "height_std", "height_range", "curvature", "roughness",
    "density", "mean_distance",
]

def preprocess(pts, lbl):
    keep = _voxel_keep(pts, CONFIG["voxel_size"])
    pts, lbl = pts[keep], (lbl[keep] if lbl is not None else None)
    keep = _sor_keep(pts, k=CONFIG["sor_k"], std_ratio=CONFIG["sor_std"])
    pts, lbl = pts[keep], (lbl[keep] if lbl is not None else None)
    keep = _cluster_keep(pts, eps=CONFIG["cluster_eps"], min_cluster_size=CONFIG["cluster_min_size"])
    pts, lbl = pts[keep], (lbl[keep] if lbl is not None else None)
    return pts, lbl

def extract_named_features(pts):
    lin, pla, sph, vert, ent, hstd, hrange, curv, rough = _geom_features(
        pts, radius=CONFIG["geom_radius"], want_linear=True, want_planar=True,
        want_sphere=True, want_vert=True, want_entropy=True, want_height_std=True,
        want_height_range=True, want_curvature=True, want_roughness=True)
    dens, mdist = _density_features(pts, k=CONFIG["density_k"],
                                    want_density=True, want_mean_distance=True)
    z = pts[:, 2]
    height = (z - z.min()) / max(z.max() - z.min(), 1e-6)
    X = np.column_stack([pts[:, 0], pts[:, 1], pts[:, 2], height,
                         lin, pla, sph, vert, ent, hstd, hrange, curv, rough,
                         dens, mdist])
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X

## Per-file loop — extract features, train a RandomForest per file, record importance

In [6]:
per_file_importance = {}   # name -> {feature: importance}
per_file_accuracy   = {}   # name -> test accuracy
skipped = []

for path in ALL_FILES:
    name = os.path.splitext(os.path.basename(path))[0]
    try:
        pts, lbl = load_pointcloud(path, return_rgb=False)
        if lbl is None:
            log.warning(f"{name}: no labels found in file -- skipping")
            skipped.append((name, "no labels"))
            continue
        pts, lbl = preprocess(pts, lbl)
        if len(pts) < CONFIG["min_points_after_preprocessing"]:
            log.warning(f"{name}: only {len(pts)} points survived preprocessing -- skipping")
            skipped.append((name, "too few points"))
            continue
        if len(np.unique(lbl)) < 2:
            log.warning(f"{name}: only one class present after preprocessing -- skipping")
            skipped.append((name, "single class"))
            continue

        X = extract_named_features(pts)
        Xtr, Xte, ytr, yte = train_test_split(
            X, lbl, test_size=CONFIG["test_size"], random_state=0, stratify=lbl)

        clf = RandomForestClassifier(
            n_estimators=CONFIG["rf_n_estimators"], max_depth=CONFIG["rf_max_depth"],
            n_jobs=-1, class_weight="balanced", random_state=0)
        clf.fit(Xtr, ytr)
        acc = clf.score(Xte, yte)

        per_file_importance[name] = dict(zip(FEATURE_NAMES, clf.feature_importances_))
        per_file_accuracy[name] = acc
        print(f"  {name:<24s} n_pts={len(pts):>7,}  test_acc={acc:.3f}")

    except Exception as e:
        log.error(f"{name}: failed -- {e}")
        skipped.append((name, str(e)))

print(f"\nProcessed {len(per_file_importance)}/{len(ALL_FILES)} files "
     f"({len(skipped)} skipped)")
if skipped:
    print("Skipped:", skipped)

  sample_data_001          n_pts= 79,140  test_acc=0.991
  sample_data_0010         n_pts= 69,315  test_acc=0.990
  sample_data_0011         n_pts= 43,916  test_acc=0.987
  sample_data_0012         n_pts=111,658  test_acc=0.990
  sample_data_0013         n_pts= 67,532  test_acc=0.980
  sample_data_0014         n_pts=103,702  test_acc=0.993
  sample_data_0015         n_pts=122,018  test_acc=0.988
  sample_data_0016         n_pts=196,059  test_acc=0.992
  sample_data_0017         n_pts=142,219  test_acc=0.991
  sample_data_0018         n_pts=240,501  test_acc=0.994
  sample_data_0019         n_pts=121,386  test_acc=0.987


  sample_data_0020         n_pts=240,289  test_acc=0.990
  sample_data_0021         n_pts= 94,272  test_acc=0.994
  sample_data_0022         n_pts=157,332  test_acc=0.989
  sample_data_0023         n_pts= 76,131  test_acc=0.993
  sample_data_0024         n_pts= 96,871  test_acc=0.989
  sample_data_0025         n_pts= 85,649  test_acc=0.991
  sample_data_0026         n_pts= 50,226  test_acc=0.995
  sample_data_0027         n_pts=108,637  test_acc=0.994
  sample_data_0028         n_pts=188,652  test_acc=0.994
  sample_data_0029         n_pts= 81,805  test_acc=0.992
  sample_data_003          n_pts=153,280  test_acc=0.989
  sample_data_0030         n_pts=102,415  test_acc=0.993
  sample_data_0031         n_pts= 79,932  test_acc=0.992
  sample_data_0032         n_pts= 66,619  test_acc=0.988
  sample_data_0033         n_pts=119,563  test_acc=0.995
  sample_data_0034         n_pts=132,114  test_acc=0.990
  sample_data_0035         n_pts= 81,270  test_acc=0.991
  sample_data_0036         n_pt

  sample_data_0020         n_pts=240,311  test_acc=0.990
  sample_data_0021         n_pts= 94,272  test_acc=0.994
  sample_data_0022         n_pts=156,509  test_acc=0.986
  sample_data_0023         n_pts= 71,933  test_acc=0.993
  sample_data_0024         n_pts= 92,911  test_acc=0.992
  sample_data_0025         n_pts= 84,954  test_acc=0.993
  sample_data_0026         n_pts= 50,226  test_acc=0.995
  sample_data_0027         n_pts=105,316  test_acc=0.992
  sample_data_0028         n_pts=188,652  test_acc=0.994
  sample_data_0029         n_pts= 81,025  test_acc=0.992
  sample_data_003          n_pts=153,280  test_acc=0.989
  sample_data_0030         n_pts=102,415  test_acc=0.993
  sample_data_0031         n_pts= 79,944  test_acc=0.992
  sample_data_0032         n_pts= 66,619  test_acc=0.988
  sample_data_0033         n_pts=119,563  test_acc=0.995
  sample_data_0034         n_pts=132,114  test_acc=0.990
  sample_data_0035         n_pts= 81,062  test_acc=0.986
  sample_data_0036         n_pt

## Build a file x feature importance table

In [7]:
imp_df = pd.DataFrame(per_file_importance).T   # rows = files, columns = features
imp_df = imp_df[FEATURE_NAMES]                  # keep a consistent column order
imp_df["test_accuracy"] = pd.Series(per_file_accuracy)
imp_df.to_csv("per_file_feature_importance.csv")
print(f"Saved per_file_feature_importance.csv  ({imp_df.shape[0]} files x {imp_df.shape[1]} columns)")
imp_df

Saved per_file_feature_importance.csv  (44 files x 16 columns)


,x,y,z,height,linearity,planarity,sphericity,verticality,eigenentropy,height_std,height_range,curvature,roughness,density,mean_distance,test_accuracy
sample_data_001,0.084355,0.125084,0.100006,0.101121,0.011769,0.028661,0.116118,0.130000,0.066818,0.022280,0.021587,0.138320,0.036831,0.010139,0.006910,0.985794
sample_data_0010,0.041509,0.097134,0.063367,0.059693,0.006760,0.022207,0.104414,0.288997,0.063996,0.053817,0.038478,0.110490,0.028918,0.012748,0.007474,0.989289
sample_data_0011,0.127754,0.082832,0.076757,0.074471,0.008555,0.021424,0.138033,0.134465,0.079968,0.012783,0.011746,0.161514,0.046736,0.013436,0.009526,0.986611
sample_data_0012,0.055061,0.130934,0.076951,0.071881,0.013272,0.039473,0.031201,0.436644,0.027426,0.046409,0.019500,0.031080,0.010048,0.005465,0.004655,0.989959
sample_data_0013,0.121526,0.366747,0.137996,0.122211,0.009606,0.008764,0.041424,0.072982,0.021456,0.012581,0.010426,0.046852,0.011824,0.008746,0.006858,0.980335
sample_data_0014,0.068061,0.117167,0.121042,0.103822,0.020955,0.024065,0.029049,0.386718,0.026951,0.037437,0.018562,0.025417,0.008060,0.007691,0.005002,0.992827
sample_data_0015,0.086845,0.133686,0.088332,0.087624,0.017849,0.018397,0.019181,0.353268,0.026800,0.073672,0.056429,0.018266,0.007329,0.007150,0.005172,0.988264
sample_data_0016,0.070865,0.056193,0.072069,0.075261,0.003645,0.003766,0.018455,0.392435,0.012494,0.149364,0.115470,0.017872,0.005873,0.003607,0.002630,0.991554
sample_data_0017,0.112458,0.130690,0.099365,0.092170,0.022388,0.020861,0.070639,0.252621,0.041394,0.023139,0.017567,0.080597,0.021886,0.008097,0.006129,0.990767
sample_data_0018,0.050417,0.039863,0.070029,0.066556,0.001656,0.002599,0.018087,0.417260,0.005548,0.163654,0.139335,0.013898,0.004303,0.004255,0.002540,0.993547


## Graph 1 — grouped bar chart, importance per feature, one bar-group per file

In [ ]:
feat_cols = FEATURE_NAMES
fig, ax = plt.subplots(figsize=(16, 7))
x = np.arange(len(feat_cols))
n_files = len(imp_df)
width = 0.8 / max(n_files, 1)

for i, (fname, row) in enumerate(imp_df[feat_cols].iterrows()):
    ax.bar(x + i * width, row.values, width=width, label=fname)

ax.set_xticks(x + width * (n_files - 1) / 2)
ax.set_xticklabels(feat_cols, rotation=45, ha="right")
ax.set_ylabel("RandomForest feature importance")
ax.set_title("Per-file feature importance -- Wood Powder Segmentation")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, ncol=1)
plt.tight_layout()
plt.savefig("per_file_feature_importance_bars.png", dpi=150, bbox_inches="tight")
plt.show()

## Graph 2 — heatmap (often easier to read with many files)

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(imp_df))))
im = ax.imshow(imp_df[feat_cols].values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(feat_cols)))
ax.set_xticklabels(feat_cols, rotation=45, ha="right")
ax.set_yticks(range(len(imp_df)))
ax.set_yticklabels(imp_df.index, fontsize=8)
ax.set_title("Per-file feature importance heatmap")
fig.colorbar(im, ax=ax, label="importance")
plt.tight_layout()
plt.savefig("per_file_feature_importance_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Graph 3 — mean importance across all files, with per-file spread (error bars = std)

In [ ]:
means = imp_df[feat_cols].mean().sort_values(ascending=False)
stds  = imp_df[feat_cols].std().reindex(means.index)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(means.index[::-1], means.values[::-1], xerr=stds.values[::-1],
       color="steelblue", ecolor="black", capsize=3)
ax.set_xlabel("Mean RandomForest importance across all files (error bar = std across files)")
ax.set_title("Overall feature importance -- mean +/- spread across files")
plt.tight_layout()
plt.savefig("overall_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Features with the LARGEST std relative to their mean (least consistent across files):")
consistency = (stds / means.replace(0, np.nan)).sort_values(ascending=False)
print(consistency.head(6))